# 04 — Model Development

**Input:** `data/processed/features.csv`  
**Outputs:** `models/random_forest.pkl`, `models/xgboost_model.pkl`, `models/lightgbm_model.pkl`

### Steps covered
1. Load features and create stratified train/test split
2. Apply SMOTE on training set only
3. Dummy classifier baseline
4. Random Forest (Candidate 1)
5. XGBoost (Candidate 2 — Primary)
6. LightGBM (Candidate 3)
7. Save all trained models

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
print('Libraries loaded.')

Libraries loaded.


## Step 1 — Load Features & Train/Test Split

`stratify=y` preserves the 43/57 failure ratio in both splits.  
`random_state=42` ensures the same split every run (reproducibility).

In [2]:
df = pd.read_csv('../data/processed/features.csv')
y = df['grid_failure_flag']
X = df.drop(columns=['grid_failure_flag'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train failure rate: {y_train.mean():.3f}')
print(f'Test  failure rate: {y_test.mean():.3f}')

Train: (40000, 40)  |  Test: (10000, 40)
Train failure rate: 0.430
Test  failure rate: 0.430


## Step 2 — SMOTE on Training Set Only

SMOTE creates synthetic minority-class (failure) examples by interpolating between existing ones.  
**Critical rule:** Apply SMOTE only to training data. Test set must represent real-world conditions.

In [3]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {dict(y_train.value_counts())}')
print(f'After  SMOTE: {dict(pd.Series(y_train_sm).value_counts())}')
print(f'Training set size after SMOTE: {len(X_train_sm)}')

Before SMOTE: {0: np.int64(22794), 1: np.int64(17206)}
After  SMOTE: {1: np.int64(22794), 0: np.int64(22794)}
Training set size after SMOTE: 45588


## Step 3 — Dummy Classifier Baseline

A model that always predicts the majority class (No Failure). Any real model must beat this.  
This is the performance floor.

In [4]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train_sm, y_train_sm)
y_pred_dummy = dummy.predict(X_test)

print('=== Baseline: Dummy Classifier (most_frequent) ===')
print(classification_report(y_test, y_pred_dummy, target_names=['No Failure', 'Failure']))

=== Baseline: Dummy Classifier (most_frequent) ===
              precision    recall  f1-score   support

  No Failure       0.57      1.00      0.73      5699
     Failure       0.00      0.00      0.00      4301

    accuracy                           0.57     10000
   macro avg       0.28      0.50      0.36     10000
weighted avg       0.32      0.57      0.41     10000



## Step 4 — Random Forest (Candidate 1)

Parallel ensemble of decision trees. Robust, handles feature interactions, gives built-in importance.  
Uses 5-fold stratified cross-validation on training set to get a stable performance estimate.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_recall_rf = cross_val_score(rf, X_train_sm, y_train_sm, cv=cv, scoring='recall', n_jobs=-1)
cv_auc_rf    = cross_val_score(rf, X_train_sm, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'RF CV Recall: {cv_recall_rf.mean():.4f} ± {cv_recall_rf.std():.4f}')
print(f'RF CV AUC-ROC: {cv_auc_rf.mean():.4f} ± {cv_auc_rf.std():.4f}')

rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)
print('\n=== Random Forest — Test Set ===')
print(classification_report(y_test, y_pred_rf, target_names=['No Failure', 'Failure']))
print(f'AUC-ROC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

RF CV Recall: 0.8232 ± 0.0050
RF CV AUC-ROC: 0.9026 ± 0.0041



=== Random Forest — Test Set ===
              precision    recall  f1-score   support

  No Failure       0.85      0.81      0.83      5699
     Failure       0.76      0.81      0.79      4301

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000

AUC-ROC: 0.8970


## Step 5 — XGBoost (Candidate 2 — Primary)

Sequential gradient boosted trees: each tree corrects errors of the previous ones.  
`scale_pos_weight = neg/pos` is XGBoost's native way to handle class imbalance — tells the model that each failure example is worth `neg/pos` non-failure examples.

In [6]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f'scale_pos_weight = {spw:.4f}')

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

cv_recall_xgb = cross_val_score(xgb, X_train_sm, y_train_sm, cv=cv, scoring='recall', n_jobs=-1)
cv_auc_xgb    = cross_val_score(xgb, X_train_sm, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'XGB CV Recall: {cv_recall_xgb.mean():.4f} ± {cv_recall_xgb.std():.4f}')
print(f'XGB CV AUC-ROC: {cv_auc_xgb.mean():.4f} ± {cv_auc_xgb.std():.4f}')

xgb.fit(X_train_sm, y_train_sm, verbose=False)
y_pred_xgb = xgb.predict(X_test)
print('\n=== XGBoost — Test Set ===')
print(classification_report(y_test, y_pred_xgb, target_names=['No Failure', 'Failure']))
print(f'AUC-ROC: {roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]):.4f}')

scale_pos_weight = 1.3248


XGB CV Recall: 0.8735 ± 0.0063
XGB CV AUC-ROC: 0.9277 ± 0.0035



=== XGBoost — Test Set ===
              precision    recall  f1-score   support

  No Failure       0.87      0.81      0.84      5699
     Failure       0.77      0.84      0.81      4301

    accuracy                           0.82     10000
   macro avg       0.82      0.83      0.82     10000
weighted avg       0.83      0.82      0.83     10000

AUC-ROC: 0.9137


## Step 6 — LightGBM (Candidate 3)

Histogram-based gradient boosting — faster than XGBoost and supports incremental retraining.  
Included because if PowerGrid wants to retrain on new asset data without full retraining, LightGBM supports that natively.

In [7]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

cv_recall_lgbm = cross_val_score(lgbm, X_train_sm, y_train_sm, cv=cv, scoring='recall', n_jobs=-1)
cv_auc_lgbm    = cross_val_score(lgbm, X_train_sm, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'LGBM CV Recall: {cv_recall_lgbm.mean():.4f} ± {cv_recall_lgbm.std():.4f}')
print(f'LGBM CV AUC-ROC: {cv_auc_lgbm.mean():.4f} ± {cv_auc_lgbm.std():.4f}')

lgbm.fit(X_train_sm, y_train_sm)
y_pred_lgbm = lgbm.predict(X_test)
print('\n=== LightGBM — Test Set ===')
print(classification_report(y_test, y_pred_lgbm, target_names=['No Failure', 'Failure']))
print(f'AUC-ROC: {roc_auc_score(y_test, lgbm.predict_proba(X_test)[:,1]):.4f}')

LGBM CV Recall: 0.8435 ± 0.0068
LGBM CV AUC-ROC: 0.9272 ± 0.0035



=== LightGBM — Test Set ===
              precision    recall  f1-score   support

  No Failure       0.85      0.84      0.85      5699
     Failure       0.79      0.81      0.80      4301

    accuracy                           0.83     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.83      0.83      0.83     10000

AUC-ROC: 0.9141


## Step 7 — CV Summary Comparison

In [8]:
summary = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LightGBM'],
    'CV Recall (mean)': [
        cv_recall_rf.mean(), cv_recall_xgb.mean(), cv_recall_lgbm.mean()
    ],
    'CV Recall (std)': [
        cv_recall_rf.std(), cv_recall_xgb.std(), cv_recall_lgbm.std()
    ],
    'CV AUC-ROC (mean)': [
        cv_auc_rf.mean(), cv_auc_xgb.mean(), cv_auc_lgbm.mean()
    ]
})
print('=== Cross-Validation Summary ===')
print(summary.to_string(index=False))

=== Cross-Validation Summary ===
        Model  CV Recall (mean)  CV Recall (std)  CV AUC-ROC (mean)
Random Forest          0.823156         0.005007           0.902640
      XGBoost          0.873476         0.006272           0.927702
     LightGBM          0.843512         0.006845           0.927225


## Step 8 — Save All Trained Models

In [9]:
os.makedirs('../models', exist_ok=True)

model_paths = {
    '../models/random_forest.pkl': rf,
    '../models/xgboost_model.pkl': xgb,
    '../models/lightgbm_model.pkl': lgbm
}

for path, model in model_paths.items():
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'Saved: {path}')

print('\nAll models saved. Proceed to Story 05 for hyperparameter tuning.')

Saved: ../models/random_forest.pkl
Saved: ../models/xgboost_model.pkl


Saved: ../models/lightgbm_model.pkl

All models saved. Proceed to Story 05 for hyperparameter tuning.
